In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Parâmetros mais usados na transpilação

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido com os seguintes requisitos.
Recomendamos usar estas versões ou versões mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Esta página descreve alguns dos parâmetros mais usados na transpilação local. Esses parâmetros são configurados como argumentos de [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) ou [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Grau de aproximação
Você pode usar o grau de aproximação para especificar o quão próximo você deseja que o circuito resultante seja do circuito desejado (de entrada). Trata-se de um número de ponto flutuante no intervalo (0,0 – 1,0), em que 0,0 representa aproximação máxima e 1,0 (padrão) representa nenhuma aproximação. Valores menores trocam precisão na saída por facilidade de execução (ou seja, menos portas). O valor padrão é 1,0.

Na síntese unitária de dois qubits (usada nas etapas iniciais de todos os níveis e na etapa de otimização com nível de otimização 3), esse valor especifica a fidelidade alvo da decomposição de saída. Ou seja, quanto de erro é introduzido quando uma representação matricial de um circuito é convertida em portas discretas. Se o grau de aproximação for um valor menor (mais aproximação), o circuito de saída da síntese diferirá mais da matriz de entrada, mas provavelmente terá menos portas (porque qualquer operação arbitrária de dois qubits pode ser decomposta perfeitamente com no máximo três portas CX) e será mais fácil de executar.

Quando o grau de aproximação é menor que 1,0, circuitos com uma ou duas portas CX podem ser sintetizados, resultando em menos erro proveniente do hardware, mas mais erro proveniente da aproximação. Como a porta CX é a mais custosa em termos de erro, pode ser benéfico reduzir a quantidade delas ao custo da fidelidade na síntese (essa técnica foi usada para aumentar o volume quântico em dispositivos IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Como exemplo, geramos um `UnitaryGate` aleatório de dois qubits que será sintetizado na etapa inicial. Definir o `approximation_degree` com um valor menor que 1,0 pode gerar um circuito aproximado. É necessário também especificar os `basis_gates` para informar ao método de síntese quais portas ele pode usar na síntese aproximada.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Isso produz uma saída `2` porque a aproximação exige menos portas CX.

<span id="seed"></span>
## Semente do gerador de números aleatórios
Algumas partes do transpilador são estocásticas, portanto execuções repetidas de transpilação podem retornar resultados diferentes. Para obter um resultado reproduzível, você pode definir a semente do gerador de números pseudoaleatórios usando o argumento `seed_transpiler`. Execuções repetidas com a mesma semente retornarão os mesmos resultados.

Exemplo:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Layout inicial
Antes da transpilação, os qubits contidos no seu circuito são qubits virtuais que não correspondem necessariamente a qubits físicos no backend alvo. Você pode especificar o mapeamento inicial de qubits virtuais para qubits físicos usando o argumento `initial_layout`. Note que o layout final de qubits pode diferir do layout inicial, pois o transpilador pode permutar qubits usando portas de swap ou outros meios.

No exemplo abaixo, construímos um layout inicial para o backend simulado [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) criando um objeto [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Nosso layout mapeia o primeiro qubit do nosso circuito para o qubit 5 de Sherbrooke, e o segundo qubit do nosso circuito para o qubit 6 de Sherbrooke. Note que qubits físicos são sempre representados por inteiros.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Além de especificar um objeto Layout, você também pode passar uma lista de inteiros, em que o $i$-ésimo elemento da lista contém o qubit físico para o qual o $i$-ésimo qubit deve ser mapeado. Por exemplo:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Você pode usar a função [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) para gerar um diagrama do grafo do dispositivo com informações de erro e com os qubits físicos identificados. Você também pode ver diagramas semelhantes na página de [Recursos de computação](https://quantum.cloud.ibm.com/computers).